In [1]:
import pandas as pd
import numpy as np
import torch_geometric
from torch_geometric.data import Data, DataLoader
import ast
import os
from itertools import combinations

d:\software\Anaconda3\envs\finnlp\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#对股票代码进行格式统一
class EXCHANGE():
    XSHG = 'XSHG'
    SSE = 'XSHG'
    SH = 'SH'
    XSHE = 'XSHE'
    SZ = 'SZ'
    SZE = 'XSHE'

def normalize_code(symbol, pre_close=None):
    """
    归一化证券代码
    
    :param symbol: 如 '1' 或 '000001'
    :param pre_close: 前收盘价（用于推断指数）
    :return: 证券代码的全称 如 '000001.SZ'
    """
    if not isinstance(symbol, str):
        return symbol

    # 处理带交易所前缀的代码（如 sz000001）
    if symbol.startswith('sz') and len(symbol) == 8:
        normalized = symbol[2:8].zfill(6)
        return f"{normalized}.{EXCHANGE.SZ}"
    elif symbol.startswith('sh') and len(symbol) == 8:
        normalized = symbol[2:8].zfill(6)
        return f"{normalized}.{EXCHANGE.SH}"

    # 尝试补全为6位代码（如 '1' → '000001'）
    if len(symbol) < 6:
        symbol = symbol.zfill(6)  # 左侧补零到6位

    # 仅处理6位代码
    if len(symbol) != 6:
        return symbol  # 非6位代码直接返回

    # 规则匹配
    if symbol.startswith('00'):
        if pre_close and pre_close > 2000:  # 推断为上证指数
            return f"{symbol}.{EXCHANGE.SH}"
        else:
            return f"{symbol}.{EXCHANGE.SZ}"
    elif symbol.startswith(('399', '159', '150', '16', '184801', '201872')):
        return f"{symbol}.{EXCHANGE.SZ}"
    elif symbol.startswith(('50', '51', '60', '688', '900')) or symbol == '751038':
        return f"{symbol}.{EXCHANGE.SH}"
    elif symbol[:3] in ['000', '001', '002', '200', '300']:
        return f"{symbol}.{EXCHANGE.SZ}"
    else:
        return f"{symbol}.{EXCHANGE.SZ}"  # 默认归属深交所（根据需求调整）

In [ ]:
news_df=pd.read_csv("../data_raw/新闻_20220328.csv")

news_df.shape

(1068, 11)

In [ ]:
news_df["stocks"] = news_df["stocks"].apply(ast.literal_eval)

In [12]:
news0328=news_df.explode('stocks')
news0328.reset_index(drop=True, inplace=True)
news0328

,Unnamed: 0,id,title,content,stocks,new_date_column,宏观,概念,行业,公司,财务
0,0,942976,Q1暴赚50亿+同环比双双翻倍！硅料龙头一季报预告亮眼，行业竞争格局不断优化？,<p><strong>财联社（上海，编辑 王舒蕾）讯</strong>，今年2月中旬以来，硅...,{'stocks_code': 'sh600089'},2022-03-28,NaN,"光伏, 新能源",NaN,"特变电工, 通威股份, 光大证券, 大全能源",NaN
1,0,942976,Q1暴赚50亿+同环比双双翻倍！硅料龙头一季报预告亮眼，行业竞争格局不断优化？,<p><strong>财联社（上海，编辑 王舒蕾）讯</strong>，今年2月中旬以来，硅...,{'stocks_code': 'sh600438'},2022-03-28,NaN,"光伏, 新能源",NaN,"特变电工, 通威股份, 光大证券, 大全能源",NaN
2,0,942976,Q1暴赚50亿+同环比双双翻倍！硅料龙头一季报预告亮眼，行业竞争格局不断优化？,<p><strong>财联社（上海，编辑 王舒蕾）讯</strong>，今年2月中旬以来，硅...,{'stocks_code': 'sh601012'},2022-03-28,NaN,"光伏, 新能源",NaN,"特变电工, 通威股份, 光大证券, 大全能源",NaN
3,0,942976,Q1暴赚50亿+同环比双双翻倍！硅料龙头一季报预告亮眼，行业竞争格局不断优化？,<p><strong>财联社（上海，编辑 王舒蕾）讯</strong>，今年2月中旬以来，硅...,{'stocks_code': 'sh688303'},2022-03-28,NaN,"光伏, 新能源",NaN,"特变电工, 通威股份, 光大证券, 大全能源",NaN
4,1,942986,山东玻纤业绩预告：预计2022年一季报营业收入变动7%至19%,财联社3月28日电，山东玻纤发布2022年一季报预增公告，去年同期净利润为15251.97万...,{'stocks_code': 'sh605006'},2022-03-28,NaN,NaN,NaN,山东玻纤,"净利润, 营业收入"
...,...,...,...,...,...,...,...,...,...,...,...
1212,1063,944055,东方中科：2021年度报告净利润17185.87万元，同比增长212.22%,财联社3月28日电，东方中科（002819）发布2021年度报告财务报告，公司2021年度报...,{'stocks_code': 'sz002819'},2022-03-28,NaN,NaN,NaN,东方中科,"每股收益, 净利润, 营业收入"
1213,1064,944056,· 华商报：宝鸡秦川机床HMC80高精度卧式加工中心 亮相国际机床展览会,"<p>4月15日,为期五天的全球机床工具行业盛会---第十六届中国国际机床展览会(CIMT2...",{'stocks_code': 'sz000837'},2022-03-28,NaN,"机器人, 军工, 新能源, 工业母机, 航空发动机","工程机械, 航天, 航空","秦川机床, 机器人",NaN
1214,1065,944057,兰石重装：科技创新助推检测服务提质增效,<p>日前，兰石重装子公司检测公司“无线应变测试系统”在高压容器残余应力检测中成功应用，得到...,{'stocks_code': 'sh603169'},2022-03-28,NaN,NaN,NaN,兰石重装,NaN
1215,1066,944058,发生了什么？164家机构蜂拥调研海尔生物,<p>财联社3月28日电，2022年3月25日，海尔生物接受来自3W Fund Manage...,{'stocks_code': 'sh688139'},2022-03-28,NaN,"物联网, 大数据, 智慧城市, 生物医药, 太阳能, 高校","疫苗, 铜","太阳能, 海尔生物","净资产收益, 净资产收益, 净利润"


In [13]:
# 定义自定义函数，提取字典中的值
def extract_stock_code(stock):
    return stock['stocks_code']

# 应用自定义函数
news0328['stocks'] = news0328['stocks'].apply(extract_stock_code)

In [14]:
news0328.head()

,Unnamed: 0,id,title,content,stocks,new_date_column,宏观,概念,行业,公司,财务
0,0,942976,Q1暴赚50亿+同环比双双翻倍！硅料龙头一季报预告亮眼，行业竞争格局不断优化？,<p><strong>财联社（上海，编辑 王舒蕾）讯</strong>，今年2月中旬以来，硅...,sh600089,2022-03-28,NaN,"光伏, 新能源",NaN,"特变电工, 通威股份, 光大证券, 大全能源",NaN
1,0,942976,Q1暴赚50亿+同环比双双翻倍！硅料龙头一季报预告亮眼，行业竞争格局不断优化？,<p><strong>财联社（上海，编辑 王舒蕾）讯</strong>，今年2月中旬以来，硅...,sh600438,2022-03-28,NaN,"光伏, 新能源",NaN,"特变电工, 通威股份, 光大证券, 大全能源",NaN
2,0,942976,Q1暴赚50亿+同环比双双翻倍！硅料龙头一季报预告亮眼，行业竞争格局不断优化？,<p><strong>财联社（上海，编辑 王舒蕾）讯</strong>，今年2月中旬以来，硅...,sh601012,2022-03-28,NaN,"光伏, 新能源",NaN,"特变电工, 通威股份, 光大证券, 大全能源",NaN
3,0,942976,Q1暴赚50亿+同环比双双翻倍！硅料龙头一季报预告亮眼，行业竞争格局不断优化？,<p><strong>财联社（上海，编辑 王舒蕾）讯</strong>，今年2月中旬以来，硅...,sh688303,2022-03-28,NaN,"光伏, 新能源",NaN,"特变电工, 通威股份, 光大证券, 大全能源",NaN
4,1,942986,山东玻纤业绩预告：预计2022年一季报营业收入变动7%至19%,财联社3月28日电，山东玻纤发布2022年一季报预增公告，去年同期净利润为15251.97万...,sh605006,2022-03-28,NaN,NaN,NaN,山东玻纤,"净利润, 营业收入"


In [33]:
news0328["stocks"]=news0328["stocks"].apply(normalize_code)
news0328.shape


(1217, 11)

In [29]:
constituents_df = pd.read_csv("../CSIA100_normalized.csv")
constituent_codes = set(constituents_df['成份券代码Constituent Code'])

# 筛选新闻数据
filtered_news = news0328[news0328['stocks'].isin(constituent_codes)]
#对结果根据ts_code进行去重
filtered_news=filtered_news.drop_duplicates(subset=['stocks'])

In [32]:
filtered_news.shape

(45, 11)